## Assignment
UCI Bank Marketing Dataset

## Analysis
 Both Logistic regression and Multi Layered Perceptron gave similar accuracy i.e. around 97.5%
 This percentage comes from the accuracy i.e. 0.975 
 This means that the problem can be solved via the basic linear model.
Also from the curve  see that both models learned the pattern properly however the Logistic Regression reduced the Loss faster and reached a slightly lower value. So Logistic regression seems more efficient for this data set.
Also Logistic Regression trained much faster than the Multi Layered Perceptron (aka MLP)

Overall Logistic regression is a better choice here because it is simpler, faster and gives same results.


In [152]:
import numpy as np
import pandas as pd
import time


# Load Dataset  bank-full.csv

In [153]:
df = pd.read_csv("bank-full-600.csv", sep=';')
X_df = df.drop("y", axis=1)   # keep as DataFrame
y = (df["y"] == "yes").astype(int).values
X_df = pd.get_dummies(X_df, drop_first=True)

# Preprocessing

In [154]:
X = X_df.astype(float).values

class StandardScalerManual:
    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)
    def transform(self, X):
        return (X - self.mean) / (self.std + 1e-8)
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

def train_test_split_manual(X, y, test_size=0.2):
    np.random.seed(42)
    idx = np.random.permutation(len(X))
    split = int(len(X)*(1-test_size))
    return X[idx[:split]], X[idx[split:]], y[idx[:split]], y[idx[split:]]

X_train, X_test, y_train, y_test = train_test_split_manual(X, y)

scaler = StandardScalerManual()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


## Logistic Regression

In [155]:
class LogisticRegression:
    def __init__(self, lr=0.01, epochs=500):
        self.lr = lr
        self.epochs = epochs

    def sigmoid(self, z):
     z = z.astype(float)
    
     result = np.zeros_like(z)
    
     # positive mask
     pos_mask = z >= 0
     neg_mask = ~pos_mask

     # safe computations
     result[pos_mask] = 1 / (1 + np.exp(-z[pos_mask]))
     result[neg_mask] = np.exp(z[neg_mask]) / (1 + np.exp(z[neg_mask]))

     return result

    def fit(self, X, y):
        self.w = np.zeros(X.shape[1])
        self.b = 0
        self.loss_history = []

        for _ in range(self.epochs):
            z = np.dot(X, self.w) + self.b
            y_pred = self.sigmoid(z)

            loss = -np.mean(y*np.log(y_pred+1e-8)+(1-y)*np.log(1-y_pred+1e-8))
            self.loss_history.append(loss)

            dw = np.dot(X.T, (y_pred-y))/len(y)
            db = np.mean(y_pred-y)

            self.w -= self.lr*dw
            self.b -= self.lr*db

    def predict(self, X):
        return (self.sigmoid(np.dot(X, self.w)+self.b) > 0.5).astype(int)


In [156]:
start = time.time()
baseline = LogisticRegression()
baseline.fit(X_train, y_train)
baseline_time = time.time() - start

baseline_acc = np.mean(baseline.predict(X_test) == y_test)
baseline_acc


np.float64(0.975)

## MLP

In [157]:
class MLP:
    def __init__(self, layers, lr=0.01, epochs=300):
        self.layers = layers
        self.lr = lr
        self.epochs = epochs

    def initialize_parameters(self):
        self.params = {}
        for i in range(len(self.layers)-1):
            self.params["W"+str(i)] = np.random.randn(self.layers[i], self.layers[i+1])*0.01
            self.params["b"+str(i)] = np.zeros((1, self.layers[i+1]))

    def relu(self, Z):
        return np.maximum(0, Z)

    def sigmoid(self, z):
     z = z.astype(float)
    
     result = np.zeros_like(z)
    
     # positive mask
     pos_mask = z >= 0
     neg_mask = ~pos_mask

     # safe computations
     result[pos_mask] = 1 / (1 + np.exp(-z[pos_mask]))
     result[neg_mask] = np.exp(z[neg_mask]) / (1 + np.exp(z[neg_mask]))

     return result

    def forward_propagation(self, X):
        self.cache = {"A0": X}
        for i in range(len(self.layers)-2):
            Z = np.dot(self.cache["A"+str(i)], self.params["W"+str(i)]) + self.params["b"+str(i)]
            self.cache["A"+str(i+1)] = self.relu(Z)
        Z = np.dot(self.cache["A"+str(len(self.layers)-2)], self.params["W"+str(len(self.layers)-2)]) + self.params["b"+str(len(self.layers)-2)]
        self.cache["A"+str(len(self.layers)-1)] = self.sigmoid(Z)
        return self.cache["A"+str(len(self.layers)-1)]

    def backward_propagation(self, y):
        m = y.shape[0]
        grads = {}
        A_final = self.cache["A"+str(len(self.layers)-1)]
        dZ = A_final - y.reshape(-1,1)

        for i in reversed(range(len(self.layers)-1)):
            A_prev = self.cache["A"+str(i)]
            grads["dW"+str(i)] = np.dot(A_prev.T, dZ)/m
            grads["db"+str(i)] = np.mean(dZ, axis=0, keepdims=True)

            if i > 0:
                dA = np.dot(dZ, self.params["W"+str(i)].T)
                dZ = dA * (A_prev > 0)
        return grads

    def fit(self, X, y):
        self.initialize_parameters()
        self.loss_history = []
        for _ in range(self.epochs):
            A = self.forward_propagation(X)
            loss = -np.mean(y*np.log(A+1e-8)+(1-y)*np.log(1-A+1e-8))
            self.loss_history.append(loss)

            grads = self.backward_propagation(y)
            for i in range(len(self.layers)-1):
                self.params["W"+str(i)] -= self.lr*grads["dW"+str(i)]
                self.params["b"+str(i)] -= self.lr*grads["db"+str(i)]

    def predict(self, X):
        return (self.forward_propagation(X) > 0.5).astype(int).flatten()


In [158]:
start = time.time()
mlp = MLP([X_train.shape[1], 16, 1])
mlp.fit(X_train, y_train)
mlp_time = time.time() - start

mlp_acc = np.mean(mlp.predict(X_test) == y_test)
mlp_acc


np.float64(0.975)

In [159]:
def get_results():
    return {
        "dataset_name": "UCI Bank Marketing",
        "n_samples": int(X.shape[0]),
        "n_features": int(X.shape[1]),
        "problem_type": "Binary Classification",
        "primary_metric": "Accuracy",
        "baseline_model": {"accuracy": float(baseline_acc), "time": baseline_time},
        "mlp_model": {"accuracy": float(mlp_acc), "time": mlp_time}
    }

get_results()


{'dataset_name': 'UCI Bank Marketing',
 'n_samples': 600,
 'n_features': 26,
 'problem_type': 'Binary Classification',
 'primary_metric': 'Accuracy',
 'baseline_model': {'accuracy': 0.975, 'time': 0.018821239471435547},
 'mlp_model': {'accuracy': 0.975, 'time': 0.08758807182312012}}

In [160]:
def confusion_matrix(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return {"TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn)}

def classification_metrics(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1)
    }


In [161]:
baseline_preds = baseline.predict(X_test)
mlp_preds = mlp.predict(X_test)


In [162]:
def get_results():
    return {
        "dataset_name": "UCI Bank Marketing",
        "n_samples": int(X.shape[0]),
        "n_features": int(X.shape[1]),
        "problem_type": "Binary Classification",
        "primary_metric": "Accuracy",

        "baseline_model": {
            "accuracy": float(np.mean(baseline_preds == y_test)),
            "time": baseline_time,
            "confusion_matrix": confusion_matrix(y_test, baseline_preds),
            "metrics": classification_metrics(y_test, baseline_preds)
        },

        "mlp_model": {
            "accuracy": float(np.mean(mlp_preds == y_test)),
            "time": mlp_time,
            "confusion_matrix": confusion_matrix(y_test, mlp_preds),
            "metrics": classification_metrics(y_test, mlp_preds)
        }
    }

get_results()


{'dataset_name': 'UCI Bank Marketing',
 'n_samples': 600,
 'n_features': 26,
 'problem_type': 'Binary Classification',
 'primary_metric': 'Accuracy',
 'baseline_model': {'accuracy': 0.975,
  'time': 0.018821239471435547,
  'confusion_matrix': {'TP': 0, 'TN': 117, 'FP': 0, 'FN': 3},
  'metrics': {'precision': 0.0, 'recall': 0.0, 'f1_score': 0.0}},
 'mlp_model': {'accuracy': 0.975,
  'time': 0.08758807182312012,
  'confusion_matrix': {'TP': 0, 'TN': 117, 'FP': 0, 'FN': 3},
  'metrics': {'precision': 0.0, 'recall': 0.0, 'f1_score': 0.0}}}

In [163]:
new_customer = {
    "age": 30,
    "job": "management",
    "marital": "single",
    "education": "tertiary",
    "default": "no",
    "balance": 8000,        
    "housing": "no",         
    "loan": "no",
    "contact": "cellular",
    "day": 15,
    "month": "aug",
    "duration": 1000,        
    "campaign": 1,
    "pdays": 10,
    "previous": 3,           
    "poutcome": "success"    
}

new_df = pd.DataFrame([new_customer])

new_df = pd.get_dummies(new_df)
new_df = new_df.reindex(columns=X_df.columns, fill_value=0)
new_X = new_df.astype(float).values
new_X = scaler.transform(new_X)

print("Logistic Prediction:", baseline.predict(new_X)[0])
print("MLP Prediction:", mlp.predict(new_X)[0])


Logistic Prediction: 1
MLP Prediction: 0


In [ ]:
import matplotlib.pyplot as plt

# Plot loss curves
plt.figure()

plt.plot(baseline.loss_history, label="Logistic Regression")
plt.plot(mlp.loss_history, label="MLP")

plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.legend()

plt.show()